# ARC-AGI-3 Local VLM Capability Check

OpenAI API 없이, Kaggle Input이나 로컬 디스크에 올린 작은 VLM으로 ARC-AGI-3 화면 이해 능력을 확인하는 노트북입니다.

권장 모델 크기: 0.5B~3B급. 예: SmolVLM 계열, Qwen2-VL 2B 계열, InternVL 1B/2B 계열처럼 `transformers`에서 로컬 로드 가능한 모델.

필수 설정: `LOCAL_VLM_MODEL_PATH`를 모델 폴더로 지정하세요. 인터넷 다운로드 없이 `local_files_only=True`로 로드합니다.

In [ ]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint

import numpy as np
from IPython.display import display, Markdown

try:
    from PIL import Image
except Exception as exc:
    Image = None
    print('PIL unavailable:', repr(exc))

IN_KAGGLE = Path('/kaggle/input').exists()
COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
ENV_DIR = next(
    (p for p in [COMP_ROOT / 'environment_files', Path.cwd() / 'environment_files', Path('/kaggle/working/agi-arc/environment_files')] if p.exists()),
    COMP_ROOT / 'environment_files',
)
REPO_ROOT = Path(os.getenv('AGI_ARC_REPO', Path.cwd()))

if IN_KAGGLE and COMP_ROOT.exists():
    wheel_dir = COMP_ROOT / 'arc_agi_3_wheels'
    if wheel_dir.exists():
        !pip install --no-index --find-links {wheel_dir} arc-agi python-dotenv -q

for candidate in [REPO_ROOT / 'src', Path.cwd() / 'src', Path('/kaggle/working/agi-arc/src')]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

# Avoid optional torchvision import paths. Some Kaggle/Pillow combinations break on PIL.ImageDraw,
# but local VLM image processors can still work through PIL directly.
os.environ.setdefault('TRANSFORMERS_NO_TORCHVISION', '1')
os.environ.setdefault('USE_TORCHVISION', '0')

MODEL_PATH = Path(os.getenv('LOCAL_VLM_MODEL_PATH', ''))
ALLOW_DOWNLOAD = os.getenv('LOCAL_VLM_ALLOW_DOWNLOAD', '0') == '1'

print('ENV_DIR =', ENV_DIR)
print('REPO_ROOT =', REPO_ROOT)
print('MODEL_PATH =', MODEL_PATH if str(MODEL_PATH) else '(not set)')
print('MODEL_PATH exists =', MODEL_PATH.exists() if str(MODEL_PATH) else False)
print('ALLOW_DOWNLOAD =', ALLOW_DOWNLOAD)

In [ ]:
# Kaggle에서 모델을 dataset으로 붙인 뒤, 예를 들어 이렇게 지정하세요.
# os.environ['LOCAL_VLM_MODEL_PATH'] = '/kaggle/input/smolvlm-500m-instruct/SmolVLM-500M-Instruct'
# os.environ['LOCAL_VLM_MODEL_PATH'] = '/kaggle/input/qwen2-vl-2b-instruct/Qwen2-VL-2B-Instruct'
# os.environ['LOCAL_VLM_MODEL_PATH'] = '/kaggle/input/internvl2-1b/InternVL2-1B'
# MODEL_PATH = Path(os.environ['LOCAL_VLM_MODEL_PATH'])

def list_possible_model_dirs(root='/kaggle/input', max_depth=3):
    root = Path(root)
    if not root.exists():
        return []
    hits = []
    for config in root.rglob('config.json'):
        rel_depth = len(config.relative_to(root).parts)
        if rel_depth <= max_depth + 1:
            hits.append(config.parent)
    return sorted(hits)

print('candidate model dirs:')
for p in list_possible_model_dirs()[:20]:
    print('-', p)

In [ ]:
ARC_PALETTE = {
    0: (0, 0, 0),
    1: (0, 116, 217),
    2: (255, 65, 54),
    3: (46, 204, 64),
    4: (255, 220, 0),
    5: (170, 80, 220),
    6: (255, 133, 27),
    7: (127, 219, 255),
    8: (135, 135, 135),
    9: (255, 255, 255),
    10: (80, 80, 80),
    11: (180, 180, 180),
    12: (128, 0, 0),
    13: (0, 128, 128),
    14: (128, 128, 0),
    15: (255, 105, 180),
}

def as_list_grid(grid):
    if grid is None:
        return []
    if hasattr(grid, 'tolist'):
        grid = grid.tolist()
    return [[int(cell) for cell in row] for row in grid]

def raw_frame_stack(raw_frame):
    return list(getattr(raw_frame, 'frame', []) or [])

def latest_grid_from_raw(raw_frame):
    frames = raw_frame_stack(raw_frame)
    return as_list_grid(frames[-1]) if frames else []

def summarize_grid(grid):
    grid = as_list_grid(grid)
    if not grid:
        return {'shape': [0, 0], 'colors': [], 'nonzero': 0}
    arr = np.asarray(grid, dtype=int)
    return {
        'shape': list(arr.shape),
        'colors': sorted(int(x) for x in np.unique(arr)),
        'nonzero': int(np.count_nonzero(arr)),
    }

def grid_to_rgb_array(grid):
    grid = as_list_grid(grid)
    if not grid:
        return np.zeros((1, 1, 3), dtype=np.uint8)
    arr = np.asarray(grid, dtype=int)
    rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.uint8)
    for value, color in ARC_PALETTE.items():
        rgb[arr == value] = color
    return rgb

def grid_to_image(grid, scale=14, draw_grid=True):
    if Image is None:
        raise RuntimeError('PIL Image is required for local VLM image input')
    rgb = grid_to_rgb_array(grid)
    h, w = rgb.shape[:2]
    img = Image.fromarray(rgb, mode='RGB').resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
    if draw_grid and scale >= 8:
        px = img.load()
        line = (35, 35, 35)
        for gx in range(0, w * scale, scale):
            for yy in range(h * scale):
                px[gx, yy] = line
        for gy in range(0, h * scale, scale):
            for xx in range(w * scale):
                px[xx, gy] = line
    return img

def display_grid(grid, scale=12):
    display(grid_to_image(grid, scale=scale, draw_grid=True))

In [ ]:
def collect_game_reset(game_id='bp35', mode='offline', render_mode=None):
    from arc_agi import Arcade, OperationMode

    arcade = Arcade(
        arc_api_key=os.getenv('ARC_API_KEY', 'test-key-123'),
        arc_base_url=os.getenv('ARC_BASE_URL', 'http://gateway:8001' if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 'https://three.arcprize.org'),
        operation_mode=OperationMode(mode.lower()),
        environments_dir=str(ENV_DIR),
        recordings_dir=str(Path('/kaggle/working/recordings') if IN_KAGGLE else Path('recordings')),
    )
    scorecard_id = arcade.create_scorecard(source_url='local-vlm-capability-check', tags=['local-vlm-capability-check'])
    wrapper = arcade.make(
        game_id,
        scorecard_id=scorecard_id,
        save_recording=False,
        include_frame_data=True,
        render_mode=render_mode,
    )
    raw = wrapper.reset()
    return arcade, wrapper, scorecard_id, raw

def close_arcade(arcade, scorecard_id):
    try:
        arcade.close_scorecard(scorecard_id)
    except Exception as exc:
        print('close_scorecard failed:', repr(exc))

def frame_metadata(raw_frame, game_id=None):
    grid = latest_grid_from_raw(raw_frame)
    return {
        'game_id': game_id or getattr(raw_frame, 'game_id', None),
        'state': getattr(getattr(raw_frame, 'state', None), 'name', str(getattr(raw_frame, 'state', None))),
        'available_actions': [getattr(a, 'name', str(a)) for a in list(getattr(raw_frame, 'available_actions', []) or [])],
        'levels_completed': getattr(raw_frame, 'levels_completed', None),
        'win_levels': getattr(raw_frame, 'win_levels', None),
        'frame_count': len(raw_frame_stack(raw_frame)),
        'latest_grid': summarize_grid(grid),
    }

GAME_ID = os.getenv('ARC_VLM_GAME_ID', 'bp35')
arcade, wrapper, scorecard_id, raw = collect_game_reset(game_id=GAME_ID, mode=os.getenv('ARC_VLM_MODE', 'offline'))
meta = frame_metadata(raw, game_id=GAME_ID)
pprint(meta)
display_grid(latest_grid_from_raw(raw), scale=12)

In [ ]:
SYSTEM_PROMPT = '''You are analyzing an ARC-AGI-3 interactive game from a grid image.
Coordinates use x,y with (0,0) at top-left.
Actions are abstract and game-specific. ACTION6 may require x,y coordinates.
Infer cautiously. Separate visual facts from hypotheses.
Explain your reasoning in natural language. Do not return JSON.
Be concrete and visual. Separate what you can directly see from what you are guessing.'''

def build_initial_prompt(meta):
    return f'''
Analyze this ARC-AGI-3 game screen.

Metadata:
{json.dumps(meta, ensure_ascii=False, indent=2)}

Please answer in these sections:

1. What I see
Describe the visible board, important objects, colors, repeated shapes, UI-like regions, obstacles, empty space, and any reference/target patterns.

2. Likely goal
Explain what the player probably needs to accomplish. If uncertain, give 2-3 possible goals and rank them.

3. Object roles
Identify which objects might be the player, goal, obstacle, movable object, reference pattern, tool, button, or paint/color source. Explain why.

4. Action hypotheses
Given the available actions, guess what each action might do. Be honest about uncertainty.

5. Best next probes
Suggest 3-5 next actions that would quickly reveal the game rules. Use exact action strings when possible, such as ACTION3 or ACTION6|x=10,y=20. For each probe, explain what result you expect and what it would teach us.

6. Risks
Mention any actions or screen regions that might be UI, no-op, reset-like, dangerous, or likely to waste steps.

7. Confidence
Give an overall confidence score from 0 to 1 and explain the main uncertainty.
'''.strip()

def build_transition_prompt(meta, action_text):
    return f'''
You see BEFORE and AFTER images from an ARC-AGI-3 game.
Action taken: {action_text}

After metadata:
{json.dumps(meta, ensure_ascii=False, indent=2)}

Explain in natural language. Do not return JSON.

Please answer in these sections:

1. Observed change
Describe exactly what changed between BEFORE and AFTER. Mention object movement, color changes, shape changes, new/removed objects, UI changes, or no visible change.

2. Likely action meaning
What does the action probably do? Is it movement, rotation, selection, painting, interaction, mode switching, or something else?

3. Did this help?
Did the change appear to move toward the likely goal? Explain why or why not.

4. Updated game hypothesis
Revise your guess about the game's goal and rules.

5. Next best probes
Suggest the next 3 actions to test. For each, say what you expect to happen.

6. Confidence
Give confidence from 0 to 1 and explain uncertainty.
'''.strip()

In [ ]:
def disable_torchvision_for_transformers():
    # Transformers may try to import torchvision while constructing AutoProcessor.
    # In some Kaggle runtimes torchvision/Pillow is partially incompatible and can
    # crash on ImageDraw or duplicate Meta kernel registration. We do not need
    # torchvision for this notebook, so make transformers treat it as unavailable.
    os.environ.setdefault('TRANSFORMERS_NO_TORCHVISION', '1')
    os.environ.setdefault('USE_TORCHVISION', '0')
    os.environ.setdefault('DISABLE_TORCHVISION', '1')
    for module_name in list(sys.modules):
        if module_name.startswith('torchvision'):
            sys.modules.pop(module_name, None)
    try:
        import transformers.utils.import_utils as transformers_import_utils
        transformers_import_utils._torchvision_available = False
        transformers_import_utils._torchvision_version = 'disabled'
    except Exception as exc:
        print('Could not patch transformers.utils.import_utils:', repr(exc))
    try:
        import transformers.utils as transformers_utils
        transformers_utils.is_torchvision_available = lambda: False
    except Exception as exc:
        print('Could not patch transformers.utils:', repr(exc))


def resolve_model_path(model_path=None):
    raw = str(model_path or os.getenv('LOCAL_VLM_MODEL_PATH', '')).strip()
    if not raw:
        candidates = list_possible_model_dirs()[:20]
        message = 'Set LOCAL_VLM_MODEL_PATH to the directory that contains config.json.'
        if candidates:
            message += '\nCandidate model directories:\n' + '\n'.join(f'- {p}' for p in candidates)
        raise ValueError(message)
    path = Path(raw).expanduser()
    if not path.exists():
        raise FileNotFoundError(f'Model path does not exist: {path}')
    if (path / 'config.json').exists():
        return path
    candidates = sorted(p.parent for p in path.rglob('config.json'))
    if len(candidates) == 1:
        print('LOCAL_VLM_MODEL_PATH pointed to a parent directory; using detected model dir:', candidates[0])
        return candidates[0]
    if candidates:
        raise ValueError(
            'LOCAL_VLM_MODEL_PATH must point to one exact model folder containing config.json. '
            f'Got parent with {len(candidates)} candidates:\n' + '\n'.join(f'- {p}' for p in candidates[:30])
        )
    visible = sorted(p.name for p in path.iterdir())[:40] if path.is_dir() else []
    raise ValueError(
        f'No config.json found under {path}. This is probably not a Hugging Face model folder. '
        f'Visible files/dirs: {visible}'
    )


class SplitVisionTextProcessor:
    def __init__(self, tokenizer, image_processor):
        self.tokenizer = tokenizer
        self.image_processor = image_processor

    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=True):
        if hasattr(self.tokenizer, 'apply_chat_template'):
            return self.tokenizer.apply_chat_template(messages, tokenize=tokenize, add_generation_prompt=add_generation_prompt)
        parts = []
        for message in messages:
            for item in message.get('content', []):
                if item.get('type') == 'text':
                    parts.append(item.get('text', ''))
                elif item.get('type') == 'image':
                    parts.append('<image>')
        if add_generation_prompt:
            parts.append('Answer:')
        return '\n'.join(parts)

    def __call__(self, text=None, images=None, return_tensors='pt'):
        if isinstance(text, list) and len(text) == 1:
            text_arg = text[0]
        else:
            text_arg = text
        encoded = self.tokenizer(text_arg, return_tensors=return_tensors)
        if images is not None and self.image_processor is not None:
            image_inputs = self.image_processor(images=images, return_tensors=return_tensors)
            encoded.update(image_inputs)
        return encoded

    def batch_decode(self, *args, **kwargs):
        return self.tokenizer.batch_decode(*args, **kwargs)


def load_processor_with_fallback(model_path, local_only):
    from transformers import AutoImageProcessor, AutoProcessor, AutoTokenizer

    try:
        return AutoProcessor.from_pretrained(
            str(model_path),
            trust_remote_code=True,
            local_files_only=local_only,
        )
    except Exception as processor_exc:
        print('AutoProcessor failed:', repr(processor_exc))
        print('Trying AutoTokenizer + AutoImageProcessor fallback...')
        tokenizer = AutoTokenizer.from_pretrained(
            str(model_path),
            trust_remote_code=True,
            local_files_only=local_only,
        )
        try:
            image_processor = AutoImageProcessor.from_pretrained(
                str(model_path),
                trust_remote_code=True,
                local_files_only=local_only,
            )
        except Exception as image_exc:
            print('AutoImageProcessor failed:', repr(image_exc))
            image_processor = None
        return SplitVisionTextProcessor(tokenizer, image_processor)


def load_local_vlm(model_path=None):
    import torch
    disable_torchvision_for_transformers()
    from transformers import AutoConfig, AutoModelForCausalLM

    model_path = resolve_model_path(model_path)
    print('resolved model_path =', model_path)

    local_only = not ALLOW_DOWNLOAD
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    processor = load_processor_with_fallback(model_path, local_only)
    common_kwargs = dict(
        trust_remote_code=True,
        local_files_only=local_only,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    if torch.cuda.is_available():
        common_kwargs['device_map'] = 'auto'

    config = AutoConfig.from_pretrained(
        str(model_path),
        trust_remote_code=True,
        local_files_only=local_only,
    )
    arch_names = list(getattr(config, 'architectures', []) or [])
    print('model_type:', getattr(config, 'model_type', None), 'architectures:', arch_names)

    model = None
    # Some transformers builds do not expose AutoModelForVision2Seq. Import it only if available.
    try:
        from transformers import AutoModelForVision2Seq  # type: ignore
        try:
            model = AutoModelForVision2Seq.from_pretrained(str(model_path), **common_kwargs)
            print('loaded with AutoModelForVision2Seq')
        except Exception as vision_exc:
            print('AutoModelForVision2Seq load failed:', repr(vision_exc))
    except Exception as import_exc:
        print('AutoModelForVision2Seq unavailable in this transformers build:', repr(import_exc))

    if model is None:
        model = AutoModelForCausalLM.from_pretrained(str(model_path), **common_kwargs)
        print('loaded with AutoModelForCausalLM')
    model.eval()
    print('loaded model:', type(model).__name__)
    return processor, model

def model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        import torch
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

processor, local_vlm = load_local_vlm()

In [ ]:
def prepare_local_vlm_inputs(processor, images, prompt):
    if not isinstance(images, list):
        images = [images]
    content = []
    for _ in images:
        content.append({'type': 'image'})
    content.append({'type': 'text', 'text': SYSTEM_PROMPT + '\n\n' + prompt})
    messages = [{'role': 'user', 'content': content}]

    if hasattr(processor, 'apply_chat_template'):
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = SYSTEM_PROMPT + '\n\n' + prompt

    try:
        return processor(text=[text], images=images, return_tensors='pt')
    except Exception:
        return processor(images=images, text=[text], return_tensors='pt')

def generate_local_vlm(processor, model, images, prompt, max_new_tokens=None):
    import torch

    max_new_tokens = int(max_new_tokens or os.getenv('LOCAL_VLM_MAX_NEW_TOKENS', '512'))
    inputs = prepare_local_vlm_inputs(processor, images, prompt)
    device = model_device(model)
    inputs = {k: (v.to(device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=getattr(processor, 'tokenizer', None).eos_token_id if getattr(processor, 'tokenizer', None) is not None else None,
        )
    input_len = inputs['input_ids'].shape[-1] if 'input_ids' in inputs else 0
    new_tokens = generated[:, input_len:] if input_len else generated
    if hasattr(processor, 'batch_decode'):
        text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    else:
        text = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip()

def ask_local_vlm_about_frame(raw_frame, game_id=None):
    grid = latest_grid_from_raw(raw_frame)
    image = grid_to_image(grid, scale=int(os.getenv('ARC_VLM_IMAGE_SCALE', '14')), draw_grid=True)
    prompt = build_initial_prompt(frame_metadata(raw_frame, game_id=game_id))
    print(prompt)
    answer = generate_local_vlm(processor, local_vlm, image, prompt)
    display(Markdown(answer))
    return answer

local_vlm_initial_answer = ask_local_vlm_about_frame(raw, game_id=GAME_ID)

In [ ]:
def parse_action(action_text):
    from arcengine import GameAction
    if '|' not in action_text:
        return GameAction.from_name(action_text), None
    name, payload_text = action_text.split('|', 1)
    payload = {}
    for part in payload_text.split(','):
        key, value = part.split('=', 1)
        payload[key.strip()] = int(value)
    return GameAction.from_name(name), payload

def step_and_compare(wrapper, action_text):
    previous_raw = getattr(step_and_compare, 'last_raw', raw)
    before_grid = latest_grid_from_raw(previous_raw)
    action, payload = parse_action(action_text)
    after_raw = wrapper.step(action, data=payload)
    step_and_compare.last_raw = after_raw
    print('action =', action_text)
    print('after meta:')
    pprint(frame_metadata(after_raw, game_id=GAME_ID))
    print('before:')
    display_grid(before_grid, scale=10)
    print('after:')
    display_grid(latest_grid_from_raw(after_raw), scale=10)
    return before_grid, after_raw

def ask_local_vlm_about_transition(before_grid, action_text, after_raw, game_id=None):
    before_image = grid_to_image(before_grid, scale=int(os.getenv('ARC_VLM_IMAGE_SCALE', '14')), draw_grid=True)
    after_image = grid_to_image(latest_grid_from_raw(after_raw), scale=int(os.getenv('ARC_VLM_IMAGE_SCALE', '14')), draw_grid=True)
    prompt = build_transition_prompt(frame_metadata(after_raw, game_id=game_id), action_text)
    print(prompt)
    answer = generate_local_vlm(processor, local_vlm, [before_image, after_image], prompt)
    display(Markdown(answer))
    return answer

# 사용 예시:
# before_grid, after_raw = step_and_compare(wrapper, 'ACTION3')
# local_vlm_transition_answer = ask_local_vlm_about_transition(before_grid, 'ACTION3', after_raw, game_id=GAME_ID)

In [ ]:
# 실험이 끝나면 scorecard를 닫습니다.
# 여러 action을 더 해보고 싶으면 이 셀은 마지막에 실행하세요.
close_arcade(arcade, scorecard_id)